In [46]:
import pandas as pd
from pathlib import Path

def fetch_data():
    cols = ["pl_name", "hostname", "discoverymethod", "pl_controv_flag", "pl_rade", "pl_bmasse", "pl_orbper", "pl_orbsmax", "pl_orbeccen", 
            "pl_insol", "pl_eqt", "st_teff", "st_rad", "st_mass", "st_lum"]  # all needed collumns
    query = "select " + ",".join(cols) + " from pscomppars" #allows selecting each collumnn from the table
    url = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
        + query.replace(" ", "+") + "&format=csv")

    path = Path("data/planets_raw.csv")
    if path.exists():
        df = pd.read_csv(path)
        print(f"Loaded {len(df)} planets from cache")
    else:
        df = pd.read_csv(url)
        path.parent.mkdir(exist_ok=True)
        df.to_csv(path, index=False)
        print(f"Downloaded {len(df)} planets from NASA")


    return df

def fill_derived(df):

    #calculates incase missing before insolation calculations
    mask = df["pl_orbsmax"].isna() & df["st_mass"].notna() & df["pl_orbper"].notna() #finds where orbdistance is absent but can be calculated
    df.loc[mask, "pl_orbsmax"] = ((df.loc[mask, "st_mass"])*(((df.loc[mask, "pl_orbper"])/365.25)**2))**(1/3) #kepler's law
    df["pl_orbsmax_derived"]=mask #states if derived

    mask = df["pl_insol"].isna() & df["st_lum"].notna() & df["pl_orbsmax"].notna() #finds where insolation is absent but it can also be calculated
    df.loc[mask, "pl_insol"] = 10 ** df.loc[mask, "st_lum"] / df.loc[mask, "pl_orbsmax"] ** 2 #update only stated rows with calculated insol
    df["pl_insol_derived"] = mask #clarifies if we derived it or not

    print(f"Filled {df['pl_orbsmax_derived'].sum()} distances, {df['pl_insol_derived'].sum()} insolations")
    return df

def elim_bad_rows(df):
    bad = (df["pl_controv_flag"] == 1) | (df["pl_rade"] <= 0) | (df["pl_orbeccen"] >= 1) #removes controversial or impossible planets
    print("Removing", bad.sum(), "planets") #Removing 58 planets
    df = df[~bad].reset_index(drop=True)
    return df

In [47]:
if __name__ == "__main__":
    df = fetch_data()
    df = fill_derived(df)
    df = elim_bad_rows(df)
    df.to_csv("data/planets_cleaned.csv", index=False)
    print("Saved", len(df), "planets")
    
'''
(df.isna().mean() * 100).sort_values(ascending=False) #finds percentage each collumn is empty for
test = pd.DataFrame({"pl_orbper": [365.25], "st_mass": [1.0], "st_lum": [0.0],
                     "pl_orbsmax": [float("nan")], "pl_insol": [float("nan")]})
fill_derived(df)
(df.isna().mean() * 100).sort_values(ascending=False) #finds percentage each collumn is empty for

print(df["pl_orbsmax_derived"].sum(), "distances filled")   #491 distances filled
print(df["pl_insol_derived"].sum(), "insolations filled")   #269 insolations filled
'''



Query: select pl_name,hostname,discoverymethod,pl_controv_flag,pl_rade,pl_bmasse,pl_orbper,pl_orbsmax,pl_orbeccen,pl_insol,pl_eqt,st_teff,st_rad,st_mass,st_lum from pscomppars
URL: https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+pl_name,hostname,discoverymethod,pl_controv_flag,pl_rade,pl_bmasse,pl_orbper,pl_orbsmax,pl_orbeccen,pl_insol,pl_eqt,st_teff,st_rad,st_mass,st_lum+from+pscomppars&format=csv
(6464, 15)
Removing 58 planets
Saved 6406 planets


'\n(df.isna().mean() * 100).sort_values(ascending=False) #finds percentage each collumn is empty for\ntest = pd.DataFrame({"pl_orbper": [365.25], "st_mass": [1.0], "st_lum": [0.0],\n                     "pl_orbsmax": [float("nan")], "pl_insol": [float("nan")]})\nfill_derived(df)\n(df.isna().mean() * 100).sort_values(ascending=False) #finds percentage each collumn is empty for\n\nprint(df["pl_orbsmax_derived"].sum(), "distances filled")   #491 distances filled\nprint(df["pl_insol_derived"].sum(), "insolations filled")   #269 insolations filled\n'

0 distances filled
0 insolations filled


Removing 58 planets
